# DB 검색 조회 속도 2차 — 벡터 저장 방식 비교 (Colab)

1차(`db_search_bench_1.ipynb`)에서 범위 안 가게가 수만 곳인 검색이 느렸고, 원인은 `vector(1024)`(약 4KB)가 TOAST에 따로 저장돼 행마다 꺼내는 비용이었습니다(`docs/db/DB_SEARCH_BENCH.md`). 같은 데이터·같은 질의로 세 가지를 비교합니다.

| 이름 | 벡터 | 저장 |
|---|---|---|
| `base` | `vector(1024)` fp32 | 지금 그대로(TOAST) |
| `plain` | `vector(1024)` fp32 | `STORAGE PLAIN`: 행 안에(페이지당 1행) |
| `half` | `halfvec(1024)` 16비트 | `STORAGE PLAIN`: 행 안에(페이지당 3행). 순위가 바뀌는지도 잼 |

**Drive에 둘 것**: 1차와 같습니다(`service_load/storesearch_v1.dump`, `service_load/bench/service_queries_v1.csv`, 서비스 모델).

**결과**: `service_load/bench/db_storage_<시각>.md`.

- 런타임: GPU(T4). 전체 약 35~45분(설치·복원 8분, 표 두 개 만들기 5~10분, 측정 20~25분).
- 1차에서 재시작 직후와 메모리에 올라온 뒤가 거의 같았으므로, 이번엔 각 표를 `pg_prewarm`으로 메모리에 올린 뒤 한 번만 잽니다.
- ⑤ 검색어 없는 경우는 벡터를 읽지 않아 뺐습니다.

In [ ]:
import os, subprocess, multiprocessing
print("CPU", multiprocessing.cpu_count(), "개")
print(subprocess.run("free -g | head -2; df -h /tmp | tail -1; nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)

In [ ]:
%%bash
# PostgreSQL 16 + PostGIS 3 + pgvector (PGDG 저장소). 1차와 같은 설정
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y curl ca-certificates lsb-release > /dev/null
install -d /usr/share/postgresql-common/pgdg
curl -s -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc https://www.postgresql.org/media/keys/ACCC4CF8.asc
echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.asc] https://apt.postgresql.org/pub/repos/apt $(lsb_release -cs)-pgdg main" > /etc/apt/sources.list.d/pgdg.list
apt-get -qq update > /dev/null
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y postgresql-16 postgresql-16-postgis-3 postgresql-16-pgvector > /dev/null
pg_ctlcluster 16 main start || true
runuser -u postgres -- psql -qc "ALTER SYSTEM SET shared_buffers = '2GB'" -c "ALTER SYSTEM SET work_mem = '64MB'" \
  -c "ALTER SYSTEM SET maintenance_work_mem = '1GB'" -c "ALTER USER postgres PASSWORD 'bench'"
pg_ctlcluster 16 main restart
runuser -u postgres -- psql -tAc "select version()"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/store-search-ai")
DUMP = ROOT / "service_load/storesearch_v1.dump"
QUERIES = ROOT / "service_load/bench/service_queries_v1.csv"
MODEL_DIR = ROOT / "runs/finetune/bge_m3_ft_mj_20261014_0743"
OUT_DIR = ROOT / "service_load/bench"
for p in [DUMP, QUERIES, MODEL_DIR]:
    assert p.exists(), f"Drive에 없음: {p}"
print(f"덤프 {DUMP.stat().st_size / 1e9:.2f}GB")

In [ ]:
%%bash
# Drive에서 로컬 디스크로 복사한 뒤 복원
set -e
cp "/content/drive/MyDrive/store-search-ai/service_load/storesearch_v1.dump" /tmp/storesearch_v1.dump
chmod 644 /tmp/storesearch_v1.dump
runuser -u postgres -- dropdb --if-exists storesearch
runuser -u postgres -- createdb storesearch
runuser -u postgres -- pg_restore -d storesearch --no-owner -j 2 /tmp/storesearch_v1.dump 2> /tmp/restore.err || true
echo "복원 경고·오류 $(grep -c ERROR /tmp/restore.err || true)건(public 스키마 등 이미 있는 것은 무시해도 됨)"; grep ERROR /tmp/restore.err | head -5 || true
runuser -u postgres -- psql -d storesearch -tAc "select version_name, status, n_total from data_version; select count(*) from store_snapshot"
rm /tmp/storesearch_v1.dump

In [ ]:
!pip install -q "psycopg[binary,pool]" sentence-transformers==5.7.0 transformers==5.18.0

In [ ]:
# 질의 벡터: 임베딩 서버와 같은 설정(fp32, 128토큰, 정규화)
import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

qs = pd.read_csv(QUERIES, dtype=str, keep_default_na=False, encoding="utf-8-sig")
qs = qs[(qs["status"] == "active") & (qs["latitude"] != "") & (qs["longitude"] != "")].reset_index(drop=True)
qs["lat"] = qs["latitude"].astype(float)
qs["lng"] = qs["longitude"].astype(float)
model = SentenceTransformer(str(MODEL_DIR), device="cuda", model_kwargs={"dtype": torch.float32})
model.max_seq_length = 128
qv = model.encode(qs["query"].tolist(), batch_size=128, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
qs["vec"] = ["[" + ",".join(f"{x:.8g}" for x in v) + "]" for v in qv]
del model
torch.cuda.empty_cache()
print(len(qs), "개 질의")

In [ ]:
import re, time, psycopg
DSN = "host=127.0.0.1 dbname=storesearch user=postgres password=bench"
conn = psycopg.connect(DSN, autocommit=True)
conn.execute("create extension if not exists pg_prewarm")
V = conn.execute("select version_id from data_version where status = 'active'").fetchone()[0]
BASE = f"store_snapshot_v{V}"
settings = {k: conn.execute(f"show {k}").fetchone()[0] for k in ["shared_buffers", "work_mem", "max_parallel_workers_per_gather"]}
pg = conn.execute("select extversion from pg_extension where extname = 'vector'").fetchone()[0]
print("버전", V, BASE, settings, "pgvector", pg)

# 질의 위치에서 가장 가까운 가게의 법정동 코드(② 시군구·시도 범위용)
qs["code"] = [conn.execute(
    f"select legal_dong_code from {BASE} where searchable "
    "order by geom <-> ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography limit 1", (r.lng, r.lat)).fetchone()[0]
    for r in qs.itertuples()]

In [ ]:
# 비교용 표 두 개: 기존 파티션을 그대로 복사하되 벡터 저장만 바꾼다. 색인은 기존 파티션과 같은 정의로 만든다
cols = [c for (c,) in conn.execute(
    "select column_name from information_schema.columns where table_name = %s order by ordinal_position", (BASE,)).fetchall()]
index_defs = [d for (d,) in conn.execute("select indexdef from pg_indexes where tablename = %s", (BASE,)).fetchall()]


def build(table, vtype):
    t = time.perf_counter()
    conn.execute(f"drop table if exists {table}")
    conn.execute(f"create table {table} (like {BASE} including defaults)")
    if vtype != "vector":
        conn.execute(f"alter table {table} alter column embedding type {vtype}(1024)")
    conn.execute(f"alter table {table} alter column embedding set storage plain")
    names = ", ".join(f'"{c}"' for c in cols)
    sel = ", ".join(f"embedding::{vtype}(1024)" if c == "embedding" else f'"{c}"' for c in cols)
    conn.execute(f"insert into {table} ({names}) select {sel} from {BASE}")
    for i, d in enumerate(index_defs):
        conn.execute(re.sub(r"INDEX \S+ ON (ONLY )?\S+", f"INDEX {table}_ix{i} ON public.{table}", d))
    conn.execute(f"vacuum analyze {table}")
    print(table, f"{time.perf_counter() - t:.0f}s")


build("bench_plain", "vector")
build("bench_half", "halfvec")

VARIANTS = [("base", BASE, "vector"), ("plain", "bench_plain", "vector"), ("half", "bench_half", "halfvec")]


def sizes(table):
    return conn.execute(
        "select pg_relation_size(c.oid), coalesce(pg_total_relation_size(nullif(c.reltoastrelid, 0)), 0), "
        "pg_indexes_size(c.oid), (select count(*) from " + table + ") , pg_relation_size(c.oid) / 8192 "
        "from pg_class c where c.relname = %s", (table,)).fetchone()


size_rows = []
for name, table, _ in VARIANTS:
    heap, toast, idx, n, pages = sizes(table)
    size_rows.append(f"| {name} | {heap / 1e9:.2f} | {toast / 1e9:.2f} | {idx / 1e9:.2f} | {n / pages:.1f} |")
print("\n".join(size_rows))

In [ ]:
# DB_TABLES.md 10-1절과 같은 쿼리. 표 이름({T})과 벡터 형({VT})만 바꾼다
TOP50 = " order by s.embedding <=> %(v)s::{VT} limit 50"
SEL = "select s.store_id, 1 - (s.embedding <=> %(v)s::{VT}) as cosine from {T} s where s.version_id = %(ver)s and s.searchable "
RADIUS = "and ST_DWithin(s.geom, ST_SetSRID(ST_MakePoint(%(lng)s, %(lat)s), 4326)::geography, %(r)s)"
PREFIX = "and s.legal_dong_code like %(p)s || '%%'"
BOX = "and s.geom && ST_MakeEnvelope(%(w)s, %(s)s, %(e)s, %(n)s, 4326)::geography"
CITY, WIDE = (0.03, 0.02), (0.15, 0.10)


def box(r, size):
    return {"w": r.lng - size[0], "s": r.lat - size[1], "e": r.lng + size[0], "n": r.lat + size[1]}


CASES = [
    ("① 반경 1km", SEL + RADIUS + TOP50, lambda r: {"r": 1000}),
    ("① 반경 10km", SEL + RADIUS + TOP50, lambda r: {"r": 10000}),
    ("② 시군구", SEL + PREFIX + TOP50, lambda r: {"p": r.code[:5]}),
    ("② 시도", SEL + PREFIX + TOP50, lambda r: {"p": r.code[:2]}),
    ("③ 지도 도시 화면", SEL + BOX + TOP50, lambda r: box(r, CITY)),
    ("③ 지도 넓은 화면", SEL + BOX + TOP50, lambda r: box(r, WIDE)),
]


def sql_for(sql, table, vtype):
    return sql.replace("{T}", table).replace("{VT}", vtype)


def params(r, extra):
    return {"v": r.vec, "ver": V, "lng": r.lng, "lat": r.lat, **extra(r)}


def timed(c, sql, p):
    t = time.perf_counter()
    c.execute(sql, p).fetchall()
    return (time.perf_counter() - t) * 1000


def pct(a, q):
    return float(np.percentile(a, q, method="higher"))


def fmt(x):
    return f"{x:.1f}"


def prewarm(table):
    # 본 표, TOAST 표, 그리고 둘의 색인 전부를 메모리에 올린다
    rels = conn.execute(
        "select c.oid from pg_class c where c.relname = %(t)s "
        "union all select c.reltoastrelid from pg_class c where c.relname = %(t)s and c.reltoastrelid <> 0",
        {"t": table}).fetchall()
    oids = [o for (o,) in rels]
    oids += [i for (i,) in conn.execute("select indexrelid from pg_index where indrelid = any(%s)", (oids,)).fetchall()]
    for oid in oids:
        conn.execute("select pg_prewarm(%s::oid::regclass)", (oid,))

In [ ]:
# 시간: 표마다 메모리에 올린 뒤 질의 전부를 한 번씩
results = {}
for name, table, vtype in VARIANTS:
    prewarm(table)
    for case, sql, extra in CASES:
        q = sql_for(sql, table, vtype)
        ms = [timed(conn, q, params(r, extra)) for r in qs.itertuples()]
        results[(name, case)] = ms
        print(f"{name:6} {case:12} p50 {fmt(pct(ms, 50)):>7} p95 {fmt(pct(ms, 95)):>7}")

In [ ]:
# halfvec 순위가 fp32와 얼마나 같은지: 반경 10km·시군구 상위 50을 base와 비교
def top_ids(case_idx, table, vtype):
    case, sql, extra = CASES[case_idx]
    q = sql_for(sql, table, vtype)
    return [[row[0] for row in conn.execute(q, params(r, extra)).fetchall()] for r in qs.itertuples()]


agree = []
for ci in [1, 2]:
    a, b = top_ids(ci, BASE, "vector"), top_ids(ci, "bench_half", "halfvec")
    top1 = np.mean([x[:1] == y[:1] for x, y in zip(a, b) if x])
    o10 = np.mean([len(set(x[:10]) & set(y[:10])) / max(1, len(x[:10])) for x, y in zip(a, b) if x])
    o50 = np.mean([len(set(x) & set(y)) / max(1, len(x)) for x, y in zip(a, b) if x])
    same10 = np.mean([x[:10] == y[:10] for x, y in zip(a, b) if x])
    agree.append(f"| {CASES[ci][0]} | {top1:.4f} | {same10:.4f} | {o10:.4f} | {o50:.4f} |")
print("\n".join(agree))

In [ ]:
# 가장 무거운 경우(같은 쿼리 30번, 첫 번 제외)와 시도 전체 EXPLAIN
v = qs["vec"].iloc[0]
sido, sido_n = conn.execute(f"select left(legal_dong_code, 2), count(*) from {BASE} where searchable "
                            "group by 1 order by 2 desc limit 1").fetchone()
lng, lat, dense_n = conn.execute(
    f"select ST_X(geom::geometry), ST_Y(geom::geometry), (select count(*) from {BASE} t where t.searchable "
    f"and ST_DWithin(t.geom, s.geom, 10000)) as n from {BASE} s tablesample system (1) "
    "where searchable and legal_dong_code like '11%%' order by n desc limit 1").fetchone()
heavy_rows, plans = [], []
for name, table, vtype in VARIANTS:
    prewarm(table)
    cases = [(f"시도 전체(코드 {sido}, {sido_n}곳)", sql_for(SEL + PREFIX + TOP50, table, vtype), {"v": v, "ver": V, "p": sido}),
             (f"서울 가장 빽빽한 곳 10km({dense_n}곳)", sql_for(SEL + RADIUS + TOP50, table, vtype),
              {"v": v, "ver": V, "lng": lng, "lat": lat, "r": 10000})]
    for case, q, p in cases:
        timed(conn, q, p)
        ms = [timed(conn, q, p) for _ in range(30)]
        heavy_rows.append(f"| {name} | {case} | {fmt(pct(ms, 50))} | {fmt(pct(ms, 95))} | {fmt(max(ms))} |")
    plan = conn.execute("explain (analyze, buffers, costs off) " + cases[0][1], cases[0][2]).fetchall()
    # 질의 벡터 숫자는 길어서 줄인다
    plans.append(f"-- {name}\n" + "\n".join(re.sub(r"'\[[^']*\]'", "'[…]'", row[0]) for row in plan))
print("\n".join(heavy_rows))
print(plans[-1])

In [ ]:
# 동시 부하: 표마다 초당 20건 × 60초, 검색 1건 = 1km + 10km(1차와 같음)
from concurrent.futures import ThreadPoolExecutor
from psycopg_pool import ConnectionPool

RATE, SECONDS = 20, 60
rows = list(qs.itertuples())
load_rows = []
for name, table, vtype in VARIANTS:
    prewarm(table)
    q1, q10 = sql_for(CASES[0][1], table, vtype), sql_for(CASES[1][1], table, vtype)
    pool = ConnectionPool(DSN, min_size=10, max_size=10, kwargs={"autocommit": True})
    pool.wait()
    lat_ms, errors = [], []

    def one(i):
        r = rows[i % len(rows)]
        t = time.perf_counter()
        try:
            with pool.connection() as c:
                c.execute(q1, params(r, CASES[0][2])).fetchall()
                c.execute(q10, params(r, CASES[1][2])).fetchall()
            lat_ms.append((time.perf_counter() - t) * 1000)
        except Exception as e:
            errors.append(repr(e))

    with ThreadPoolExecutor(32) as ex:
        start = time.perf_counter()
        for i in range(RATE * SECONDS):
            delay = start + i / RATE - time.perf_counter()
            if delay > 0:
                time.sleep(delay)
            ex.submit(one, i)
    pool.close()
    load_rows.append(f"| {name} | {len(lat_ms)} | {len(errors)} | {fmt(pct(lat_ms, 50))} | {fmt(pct(lat_ms, 95))} | "
                     f"{fmt(pct(lat_ms, 99))} | {fmt(max(lat_ms))} |")
    print(load_rows[-1], errors[:2])

In [ ]:
names = [n for n, _, _ in VARIANTS]
report = [f"# DB 검색 조회 속도 2차: 벡터 저장 방식 비교 — Colab ({time.strftime('%Y-%m-%d %H:%M')} UTC)\n",
          f"- 환경: Colab CPU {multiprocessing.cpu_count()}개, PostgreSQL 16, pgvector {pg}, shared_buffers {settings['shared_buffers']}, "
          f"work_mem {settings['work_mem']}, max_parallel_workers_per_gather {settings['max_parallel_workers_per_gather']}",
          f"- 데이터 버전 {V}, 질의 {len(qs)}개. 표마다 pg_prewarm 뒤 한 번씩. 시간 = 결과 행을 다 받을 때까지(ms)",
          "- base = vector(1024) TOAST(지금), plain = vector(1024) STORAGE PLAIN, half = halfvec(1024) STORAGE PLAIN\n",
          "## 크기(GB)\n\n| 표 | 본 행 | TOAST | 색인 | 페이지당 행 |\n|---|---|---|---|---|", *size_rows,
          "\n## p95(ms) — 기준 150\n\n| 경우 | " + " | ".join(names) + " |\n|---|" + "---|" * len(names)]
for case, _, _ in CASES:
    report.append(f"| {case} | " + " | ".join(fmt(pct(results[(n, case)], 95)) for n in names) + " |")
report.append("\n## p50(ms)\n\n| 경우 | " + " | ".join(names) + " |\n|---|" + "---|" * len(names))
for case, _, _ in CASES:
    report.append(f"| {case} | " + " | ".join(fmt(pct(results[(n, case)], 50)) for n in names) + " |")
report += ["\n## halfvec 순위가 fp32와 같은 비율(질의 평균)\n",
           "| 경우 | 1위 같음 | 상위 10 순서까지 같음 | 상위 10 겹침 | 상위 50 겹침 |\n|---|---|---|---|---|", *agree,
           f"\n## 가장 무거운 경우(질의 '{qs['query'].iloc[0]}', 30번)\n",
           "| 표 | 경우 | p50 | p95 | 최대 |\n|---|---|---|---|---|", *heavy_rows,
           "\n## 시도 전체 EXPLAIN\n\n```\n" + "\n\n".join(plans) + "\n```",
           f"\n## 동시 부하: 초당 {RATE}건 × {SECONDS}초(검색 1건 = 1km + 10km)\n",
           "| 표 | 완료 | 오류 | p50 | p95 | p99 | 최대 |\n|---|---|---|---|---|---|---|", *load_rows]
out = OUT_DIR / f"db_storage_{time.strftime('%Y%m%d_%H%M')}.md"
out.write_text("\n".join(report) + "\n", encoding="utf-8")
print(out)
print("\n".join(report))